In [3]:
import pandas as pd
import numpy as np
import joblib

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

# ==========================================
# IMPORT YOUR ENVIRONMENT
# ==========================================

from trading_env import TradingEnv

# ==========================================
# LOAD TRAIN DATA
# ==========================================

train_df = pd.read_parquet(
    "../data/processed/train.parquet"
)

print("Train rows:", len(train_df))

# ==========================================
# FEATURE LIST
# ==========================================

feature_columns = [
    "return",
    "log_return",
    "high_low_spread",
    "open_close_spread",

    "ema20_ema50_ratio",
    "ema50_ema200_ratio",

    "rsi",

    "macd",
    "macd_signal",
    "macd_hist",

    "atr",

    "volatility_24h",
    "volatility_168h",

    "volume_ratio",
    "volume_change",

    "bb_width",
    "bb_position",

    "price_vs_ema50",
    "price_vs_ema200"
]

# ==========================================
# CREATE ENVIRONMENT
# ==========================================

env = TradingEnv(
    df=train_df,
    feature_columns=feature_columns,
    initial_balance=10000,
    trading_fee=0.001
)

env = Monitor(env)

# ==========================================
# PPO MODEL
# ==========================================

model = PPO(
    policy="MlpPolicy",

    env=env,

    learning_rate=3e-4,

    gamma=0.99,

    n_steps=2048,

    batch_size=256,

    n_epochs=10,

    gae_lambda=0.95,

    clip_range=0.2,

    ent_coef=0.01,

    vf_coef=0.5,

    max_grad_norm=0.5,

    verbose=1,

    tensorboard_log="./tensorboard_logs/"
)

# ==========================================
# TRAIN
# ==========================================

TOTAL_TIMESTEPS = 500_000

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    progress_bar=True
)

# ==========================================
# SAVE MODEL
# ==========================================

model.save(
    "../models/ppo_btc"
)

print("\nModel Saved")

# ==========================================
# SAVE FEATURES
# ==========================================

joblib.dump(
    feature_columns,
    "../models/feature_columns.pkl"
)

print("Feature List Saved")

# ==========================================
# QUICK TRAINING EVALUATION
# ==========================================

obs, _ = env.reset()

portfolio_values = []

actions_taken = []

done = False

while not done:

    action, _ = model.predict(
        obs,
        deterministic=True
    )

    obs, reward, done, truncated, info = env.step(action)

    actions_taken.append(action)

    portfolio_values.append(
        info["portfolio_value"]
    )

# ==========================================
# RESULTS
# ==========================================

final_portfolio = portfolio_values[-1]

print("\n==========================")
print("Training Evaluation")
print("==========================")

print(
    f"Final Portfolio: "
    f"${final_portfolio:,.2f}"
)

print(
    f"Total Trades: "
    f"{info['trade_count']}"
)

print(
    f"Return: "
    f"{(final_portfolio/10000-1)*100:.2f}%"
)

# ==========================================
# SAVE PORTFOLIO CURVE
# ==========================================

portfolio_df = pd.DataFrame({
    "portfolio_value":
    portfolio_values
})

portfolio_df.to_csv(
    "../models/train_portfolio_curve.csv",
    index=False
)

print(
    "Portfolio Curve Saved"
)

ModuleNotFoundError: No module named 'trading_env'

In [2]:
%pip install stable_baselines3

  Using cached stable_baselines3-2.9.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached torch-2.12.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached cuda_bindings-13.3.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.5 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.29.7-py3-none